# NB_dv_business_vault — PIT & Bridge Tables

Generates the business vault layer on top of the raw vault:
- **PIT tables**: Point-in-Time snapshots aligning satellite versions to a date spine
- **Bridge tables**: Multi-hop join shortcuts across hub/link chains

Run this notebook daily (or on demand) after hub/link/satellite loads complete.

In [ ]:
%run ../helpers/NB_catalog_helpers
%run ./NB_dv_metadata

In [ ]:
from datetime import date

dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA", "vault", "Vault schema")
dbutils.widgets.text("MODEL_PATH", "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("SNAPSHOT_DATE", str(date.today()), "Snapshot date (YYYY-MM-DD)")

CATALOG       = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA  = dbutils.widgets.get("VAULT_SCHEMA")
MODEL_PATH    = dbutils.widgets.get("MODEL_PATH")
SNAPSHOT_DATE = dbutils.widgets.get("SNAPSHOT_DATE")

model = load_model(MODEL_PATH)
print(f"Snapshot date: {SNAPSHOT_DATE}")
print(f"PIT tables: {len(model.get('pit_tables', []))}")
print(f"Bridge tables: {len(model.get('bridge_tables', []))}")

In [ ]:
# ── PIT Tables ──────────────────────────────────────────────────────────────
# For each PIT table: find the latest satellite record version as of SNAPSHOT_DATE
# and write a snapshot row for each hub key.

from pyspark.sql import functions as F, Window

for pit_cfg in model.get('pit_tables', []):
    if not pit_cfg.get('enabled', True):
        print(f"  Skipping disabled PIT: {pit_cfg['name']}")
        continue

    pit_name   = pit_cfg['name']
    hub_name   = pit_cfg['hub']
    sat_names  = pit_cfg['satellites']
    hk_col     = f"HK_{hub_name[4:]}"
    target_tbl = f"{CATALOG}.{VAULT_SCHEMA}.{pit_name.lower()}"

    # Ensure PIT table exists
    create_pit_table(pit_cfg)

    # Start with hub keys as the spine
    hub_tbl  = f"{CATALOG}.{VAULT_SCHEMA}.{hub_name.lower()}"
    spine_df = spark.table(hub_tbl).select(hk_col).distinct()
    spine_df = spine_df.withColumn('SNAPSHOT_DATE', F.lit(SNAPSHOT_DATE).cast('date'))
    spine_df = spine_df.withColumn('LOAD_DATE', F.current_timestamp())

    # For each satellite: find max LOAD_DATE <= SNAPSHOT_DATE
    for sat_name in sat_names:
        sat_tbl = f"{CATALOG}.{VAULT_SCHEMA}.{sat_name.lower()}"
        try:
            sat_df = (
                spark.table(sat_tbl)
                .filter(F.col('LOAD_DATE').cast('date') <= F.lit(SNAPSHOT_DATE))
            )
            w = Window.partitionBy(hk_col).orderBy(F.col('LOAD_DATE').desc())
            sat_latest = (
                sat_df
                .withColumn('_rn', F.row_number().over(w))
                .filter(F.col('_rn') == 1)
                .select(hk_col,
                        F.col('LOAD_DATE').alias(f'{sat_name}_LDTS'),
                        F.col('DIFF_HK').alias(f'{sat_name}_HK'))
            )
            spine_df = spine_df.join(sat_latest, on=hk_col, how='left')
        except Exception as e:
            print(f"    Warning: could not join {sat_name}: {e}")

    # Delete existing snapshot for this date, then insert
    try:
        spark.sql(f"DELETE FROM {target_tbl} WHERE SNAPSHOT_DATE = '{SNAPSHOT_DATE}'")
    except Exception:
        pass
    spine_df.write.format('delta').mode('append').saveAsTable(target_tbl)
    print(f"  {pit_name}: {spine_df.count():,} snapshot rows written for {SNAPSHOT_DATE}")

In [ ]:
# ── Bridge Tables ────────────────────────────────────────────────────────────
# Build multi-hop join chains from the path definition.
# Each path alternates: HUB → LINK → HUB → LINK → HUB ...

for brg_cfg in model.get('bridge_tables', []):
    if not brg_cfg.get('enabled', True):
        print(f"  Skipping disabled bridge: {brg_cfg['name']}")
        continue

    brg_name   = brg_cfg['name']
    path       = brg_cfg['path']
    target_tbl = f"{CATALOG}.{VAULT_SCHEMA}.{brg_name.lower()}"

    # Ensure bridge table exists
    create_bridge_table(brg_cfg)

    # Start with first hub
    first_hub = path[0]
    first_hk  = f"HK_{first_hub[4:]}"
    result_df = spark.table(f"{CATALOG}.{VAULT_SCHEMA}.{first_hub.lower()}").select(first_hk)

    # Walk the path: join each link then the next hub
    i = 1
    while i < len(path) - 1:
        link_name = path[i]
        next_hub  = path[i + 1]
        link_hk   = f"HK_{link_name[4:]}"
        src_hk    = f"HK_{path[i-1][4:]}"
        tgt_hk    = f"HK_{next_hub[4:]}"

        link_df = spark.table(f"{CATALOG}.{VAULT_SCHEMA}.{link_name.lower()}")
        result_df = result_df.join(link_df, on=src_hk, how='inner').drop(link_hk)
        i += 2

    # Add load date and write
    result_df = result_df.withColumn('LOAD_DATE', F.current_timestamp())
    result_df.write.format('delta').mode('overwrite').saveAsTable(target_tbl)
    print(f"  {brg_name}: {result_df.count():,} rows written to {target_tbl}")

print('\nBusiness vault refresh complete.')